# NiyamTrace-X — Local Open-Weight External Benchmark Closure

This is the **API-free** closure notebook.

It downloads open-weight models from Hugging Face, starts **one model at a time** through a local vLLM OpenAI-compatible server, checks tool calling, runs the external benchmarks, unloads the model, and continues.

## Default model queue

| Family | Model | Role |
|---|---|---|
| Granite | `ibm-granite/granite-3.1-2b-instruct` | small independent family |
| Qwen | `Qwen/Qwen2.5-3B-Instruct` | small Qwen baseline |
| Qwen3 | `Qwen/Qwen3-1.7B` | newer Qwen tool-use variant |
| Phi | `microsoft/Phi-4-mini-instruct` | independent Microsoft family |
| Mistral | `mistralai/Ministral-3-3B-Instruct-2512` | current small agentic Mistral family |
| Qwen | `Qwen/Qwen2.5-7B-Instruct-AWQ` | quantized larger Qwen stress test |
| Mistral | `mistralai/Mistral-7B-Instruct-v0.3` | high-memory optional model |

The notebook automatically skips a model if:
- the GPU is too small,
- download fails,
- vLLM cannot load it,
- the local server fails,
- or automatic tool calling fails.

## External benchmarks

1. **BFCL-v4**
2. **AgentDojo**
3. **τ³ / tau2-bench**

## Default workload

`MODE="CLOSURE"`:
- BFCL: 20 cases/model
- AgentDojo: 1 task × 1 injection over 4 suites/model
- τ³: 2 tasks × 3 domains/model

Set `MODE="SMOKE"` for a short integration check or `MODE="FULL"` for the largest run.

At the end the notebook creates and downloads:

`NTX_LOCAL_MODELS_EXTERNAL_CLOSURE_RESULTS.zip`

Model weights are **not** placed in this ZIP.

## Recovery fix in this version

The previous T4 run never reached inference because vLLM was installed into
Colab's base Python environment and upgraded PyTorch while the preinstalled
TorchAudio remained compiled for another CUDA version.

This fixed notebook creates a **clean Python 3.12 vLLM environment with uv**
and launches every local model using that interpreter. Colab's base
Torch/TorchAudio packages are not reused by the vLLM server.


In [ ]:
# CELL 1 — CONFIGURATION
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, re, time, random, hashlib, zipfile, shutil, subprocess, signal, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED=42
random.seed(SEED)
np.random.seed(SEED)

MODE=os.getenv("NTX_LOCAL_MODE","CLOSURE").upper()
assert MODE in {"SMOKE","CLOSURE","FULL"}

BASE=Path("/content/NTX_LOCAL_EXTERNAL_CLOSURE")
WORK=BASE/"work"
MODELS=BASE/"models"
RAW=BASE/"raw"
RESULTS=BASE/"results"
LOGS=BASE/"logs"
PAPER=BASE/"paper_integration"
ARCH=BASE/"archives"
for p in [BASE,WORK,MODELS,RAW,RESULTS,LOGS,PAPER,ARCH]:
    p.mkdir(parents=True,exist_ok=True)

CFG={
    "SMOKE":{
        "bfcl_limit":3,
        "dojo_suites":["banking"],
        "dojo_user_tasks":["user_task_0"],
        "dojo_injection_tasks":["injection_task_0"],
        "tau_domains":["airline"],
        "tau_tasks":1,
        "tau_steps":16,
        "max_model_len":4096,
    },
    "CLOSURE":{
        "bfcl_limit":20,
        "dojo_suites":["banking","workspace","travel","slack"],
        "dojo_user_tasks":["user_task_0"],
        "dojo_injection_tasks":["injection_task_0"],
        "tau_domains":["airline","retail","telecom"],
        "tau_tasks":2,
        "tau_steps":32,
        "max_model_len":8192,
    },
    "FULL":{
        "bfcl_limit":None,
        "dojo_suites":["banking","workspace","travel","slack"],
        "dojo_user_tasks":None,
        "dojo_injection_tasks":None,
        "tau_domains":["airline","retail","telecom"],
        "tau_tasks":None,
        "tau_steps":64,
        "max_model_len":16384,
    }
}[MODE]

# Keep downloaded model weights after each run?
# False saves Colab disk; Hugging Face may still keep some cache metadata.
KEEP_MODEL_WEIGHTS=False

# Set to an integer to stop after N successful model preflights.
# None = try every GPU-eligible model.
MAX_SUCCESSFUL_MODELS=None

PORT=8000
LOCAL_BASE=f"http://127.0.0.1:{PORT}/v1"
LOCAL_KEY="EMPTY"

print("MODE:",MODE)
print(json.dumps(CFG,indent=2))

In [ ]:
# CELL 2 — INSTALL LOCAL INFERENCE + UTILITIES (FIXED)
# Root-cause fix:
# vLLM is installed in a clean Python 3.12 uv environment instead of
# Colab's base Python environment. This prevents stale Colab TorchAudio/
# TorchVision packages from being imported against a different CUDA build.

def sh(cmd,cwd=None,env=None,timeout=None):
    return subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        env=env,
        capture_output=True,
        text=True,
        errors="replace",
        timeout=timeout,
    )

def save_log(name,p,cmd=None):
    text=""
    if cmd:
        text+="COMMAND\n"+" ".join(map(str,cmd))+"\n\n"
    text+="STDOUT\n"+(p.stdout or "")+"\n\nSTDERR\n"+(p.stderr or "")
    (LOGS/name).write_text(text,errors="ignore")

def ensure_base_package(import_name,pip_spec=None):
    try:
        __import__(import_name)
        return
    except Exception:
        pass
    spec=pip_spec or import_name
    p=sh([sys.executable,"-m","pip","install","-q","-U",spec])
    save_log(f"install_{import_name}.log",p)
    if p.returncode:
        print(p.stderr[-4000:])
        raise RuntimeError(f"Could not install {spec}")

# uv first.
if shutil.which("uv") is None:
    p=sh([sys.executable,"-m","pip","install","-q","-U","uv"])
    save_log("install_uv.log",p)
    if p.returncode:
        raise RuntimeError("uv installation failed")

# Lightweight clients remain in base runtime.
ensure_base_package("openai","openai")
ensure_base_package("huggingface_hub","huggingface_hub")
ensure_base_package("psutil","psutil")

from openai import OpenAI
from huggingface_hub import snapshot_download, HfApi

# ------------------------------------------------------------
# Isolated vLLM environment
# ------------------------------------------------------------

VLLM_ENV=WORK/"vllm_py312"
VLLM_PY=VLLM_ENV/"bin"/"python"

# Managed Python 3.12 is the stable isolated runtime.
p=sh(["uv","python","install","3.12"])
save_log("vllm_python312_install.log",p)

if not VLLM_ENV.exists():
    p=sh([
        "uv","venv",VLLM_ENV,
        "--python","3.12",
        "--seed",
        "--managed-python",
    ])
    save_log("vllm_venv_create.log",p)
    if p.returncode:
        raise RuntimeError("Could not create isolated vLLM Python 3.12 environment.")

# Install vLLM with uv selecting a CUDA/PyTorch backend compatible with
# the runtime driver. This does NOT mutate Colab's base torch/torchaudio.
p=sh([
    "uv","pip","install",
    "--python",VLLM_PY,
    "-U",
    "vllm",
    "--torch-backend=auto",
])
save_log("vllm_isolated_install.log",p)
if p.returncode:
    print(p.stderr[-6000:])
    raise RuntimeError("Isolated vLLM installation failed.")

# Critical verification: vLLM + torch import in the exact interpreter
# that will launch the API server. Also confirm a stale torchaudio package
# is not visible in this isolated environment.
verify_code = r"""
import importlib.util, json, torch, vllm
print(json.dumps({
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "vllm": vllm.__version__,
    "torchaudio_visible": importlib.util.find_spec("torchaudio") is not None,
    "torchvision_visible": importlib.util.find_spec("torchvision") is not None,
}))
"""

v=sh([VLLM_PY,"-c",verify_code])
save_log("vllm_isolated_verify.log",v,[VLLM_PY,"-c","<verification>"])
if v.returncode:
    print(v.stderr[-6000:])
    raise RuntimeError("vLLM isolated runtime verification failed.")

print("Isolated vLLM runtime:")
print(v.stdout.strip())

# Record environment provenance.
fr=sh(["uv","pip","freeze","--python",VLLM_PY])
(LOGS/"vllm_isolated_freeze.txt").write_text(fr.stdout or "")

print("✅ vLLM is isolated from the Colab base environment.")


In [ ]:
# CELL 2B — FAIL-FAST LOCAL vLLM RUNTIME DIAGNOSTIC
# This must pass BEFORE any model download begins.

diag = sh([
    VLLM_PY,
    "-c",
    (
        "import torch,vllm,importlib.util;"
        "print('TORCH',torch.__version__);"
        "print('CUDA',torch.version.cuda);"
        "print('VLLM',vllm.__version__);"
        "print('TORCHAUDIO_VISIBLE',importlib.util.find_spec('torchaudio') is not None)"
    ),
])

print(diag.stdout)
if diag.returncode:
    print(diag.stderr)
    raise RuntimeError("Isolated vLLM runtime diagnostic failed.")

if "TORCH " not in diag.stdout or "VLLM " not in diag.stdout:
    raise RuntimeError("Unexpected vLLM diagnostic output.")

print("✅ Runtime diagnostic passed. Model downloads may proceed.")


In [ ]:
# CELL 3 — GPU / DISK DETECTION + MODEL REGISTRY

def gpu_info():
    p=sh([
        "nvidia-smi",
        "--query-gpu=name,memory.total",
        "--format=csv,noheader,nounits",
    ])
    if p.returncode:
        raise RuntimeError("No NVIDIA GPU detected. In Colab: Runtime > Change runtime type > GPU.")
    line=p.stdout.strip().splitlines()[0]
    name,mem=line.rsplit(",",1)
    return name.strip(),float(mem.strip())/1024

GPU_NAME,VRAM_GB=gpu_info()
DISK_FREE_GB=shutil.disk_usage("/content").free/(1024**3)

print("GPU:",GPU_NAME)
print(f"VRAM: {VRAM_GB:.1f} GiB")
print(f"Disk free: {DISK_FREE_GB:.1f} GiB")

# tool_parser values are current vLLM parser names.
MODEL_REGISTRY=[
    {
        "slug":"granite31_2b",
        "family":"Granite",
        "repo":"ibm-granite/granite-3.1-2b-instruct",
        "tool_parser":"granite",
        "min_vram_gb":7.0,
        "extra_server_args":[],
        "trust_remote_code":False,
    },
    {
        "slug":"qwen25_3b",
        "family":"Qwen",
        "repo":"Qwen/Qwen2.5-3B-Instruct",
        "tool_parser":"hermes",
        "min_vram_gb":8.0,
        "extra_server_args":[],
        "trust_remote_code":False,
    },
    {
        "slug":"qwen3_1_7b",
        "family":"Qwen3",
        "repo":"Qwen/Qwen3-1.7B",
        "tool_parser":"hermes",
        "min_vram_gb":7.0,
        "extra_server_args":[
            "--reasoning-parser","qwen3",
            "--default-chat-template-kwargs",'{"enable_thinking": false}',
        ],
        "trust_remote_code":False,
    },
    {
        "slug":"phi4_mini",
        "family":"Phi",
        "repo":"microsoft/Phi-4-mini-instruct",
        "tool_parser":"phi4_mini_json",
        "min_vram_gb":11.0,
        "extra_server_args":["--trust-remote-code"],
        "trust_remote_code":True,
    },
    {
        "slug":"ministral3_3b",
        "family":"Mistral",
        "repo":"mistralai/Ministral-3-3B-Instruct-2512",
        "tool_parser":"mistral",
        "min_vram_gb":11.0,
        "extra_server_args":["--language-model-only"],
        "trust_remote_code":False,
    },
    {
        "slug":"qwen25_7b_awq",
        "family":"Qwen",
        "repo":"Qwen/Qwen2.5-7B-Instruct-AWQ",
        "tool_parser":"hermes",
        "min_vram_gb":10.0,
        "extra_server_args":[],
        "trust_remote_code":False,
    },
    {
        "slug":"mistral7b_v03",
        "family":"Mistral",
        "repo":"mistralai/Mistral-7B-Instruct-v0.3",
        "tool_parser":"mistral",
        "min_vram_gb":19.0,
        "extra_server_args":[],
        "trust_remote_code":False,
    },
]

registry_df=pd.DataFrame(MODEL_REGISTRY)
registry_df["gpu_eligible"]=registry_df.min_vram_gb<=VRAM_GB
display(registry_df[["slug","family","repo","tool_parser","min_vram_gb","gpu_eligible"]])

registry_df.to_csv(RESULTS/"00_model_registry.csv",index=False)

In [ ]:
# CELL 4 — SET UP ALL THREE EXTERNAL BENCHMARKS ONCE

def clone_once(url,dest):
    dest=Path(dest)
    if not dest.exists():
        p=sh(["git","clone","--depth","1",url,dest])
        save_log("clone_"+dest.name+".log",p)
        if p.returncode:
            raise RuntimeError(f"Clone failed: {url}")
    return sh(["git","-C",dest,"rev-parse","HEAD"]).stdout.strip()

# ---------- BFCL / EvalScope isolated Python 3.11 ----------
BFENV=WORK/"bfcl_env"
sh(["uv","python","install","3.11"])
if not BFENV.exists():
    p=sh(["uv","venv",BFENV,"--python","3.11"])
    save_log("bfcl_venv.log",p)
    if p.returncode:
        raise RuntimeError("BFCL venv creation failed")
BFPY=BFENV/"bin"/"python"
p=sh(["uv","pip","install","--python",BFPY,"-U","evalscope[bfcl]"])
save_log("bfcl_install.log",p)
if p.returncode:
    raise RuntimeError("BFCL/EvalScope install failed")
v=sh([BFPY,"-c","from evalscope import run_task; from evalscope.config import TaskConfig; print('BFCL_READY')"])
if v.returncode or "BFCL_READY" not in v.stdout:
    raise RuntimeError("BFCL environment verification failed")

# ---------- AgentDojo ----------
DOJO=WORK/"agentdojo"
DOJO_COMMIT=clone_once("https://github.com/ethz-spylab/agentdojo.git",DOJO)
p=sh(["uv","sync"],cwd=DOJO)
save_log("dojo_sync.log",p)
if p.returncode:
    raise RuntimeError("AgentDojo uv sync failed")
h=sh(["uv","run","python","-m","agentdojo.scripts.benchmark","--help"],cwd=DOJO)
save_log("dojo_help.log",h)
ht=(h.stdout or "")+(h.stderr or "")
for token in ["openai-compatible","--model-id","--force-rerun"]:
    if token not in ht:
        raise RuntimeError(f"AgentDojo CLI missing {token}")

# ---------- tau2 / tau3 ----------
TAU=WORK/"tau2-bench"
TAU_COMMIT=clone_once("https://github.com/sierra-research/tau2-bench.git",TAU)
p=sh(["uv","sync"],cwd=TAU)
save_log("tau_sync.log",p)
if p.returncode:
    raise RuntimeError("tau2 uv sync failed")
TAUPY=TAU/".venv"/"bin"/"python"
p=sh(["uv","pip","install","--python",TAUPY,"websockets","soundfile"],cwd=TAU)
save_log("tau_extra_deps.log",p)
v=sh([TAUPY,"-c","import websockets,soundfile;print('TAU_READY')"],cwd=TAU)
if v.returncode or "TAU_READY" not in v.stdout:
    raise RuntimeError("tau2 dependency verification failed")

BENCHMARK_VERSIONS={
    "agentdojo_commit":DOJO_COMMIT,
    "tau2_commit":TAU_COMMIT,
}
(Path(RESULTS/"01_benchmark_versions.json")
 .write_text(json.dumps(BENCHMARK_VERSIONS,indent=2)))

print("BFCL, AgentDojo, and tau2 environments ready.")

In [ ]:
# CELL 5 — MODEL DOWNLOAD + LOCAL vLLM SERVER LIFECYCLE

SERVER=None

def remote_model_info(repo):
    api=HfApi()
    info=api.model_info(repo,files_metadata=True)
    size=sum((getattr(s,"size",0) or 0) for s in info.siblings)/(1024**3)
    return info.sha,size

def download_model(spec):
    revision,size_gb=remote_model_info(spec["repo"])
    free=shutil.disk_usage("/content").free/(1024**3)
    # Need some headroom for benchmark outputs and extraction/cache.
    required=max(5.0,size_gb*1.20+3.0)
    if free<required:
        raise RuntimeError(
            f"Not enough disk for {spec['repo']}: "
            f"need about {required:.1f} GiB, have {free:.1f} GiB"
        )
    print(f"Downloading {spec['repo']} ({size_gb:.1f} GiB remote files)...")
    path=snapshot_download(
        repo_id=spec["repo"],
        revision=revision,
        cache_dir=str(MODELS/"hf_cache"),
    )
    return Path(path),revision,size_gb

def stop_server():
    global SERVER
    if SERVER is not None:
        try:
            SERVER.terminate()
            SERVER.wait(timeout=25)
        except Exception:
            try:
                SERVER.kill()
            except Exception:
                pass
        SERVER=None
    # Kill orphan vLLM API workers if any.
    subprocess.run(
        ["pkill","-f","vllm.entrypoints.openai.api_server"],
        stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL
    )
    time.sleep(5)
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass

def start_server(spec,model_path):
    global SERVER
    stop_server()

    alias=spec["slug"]
    logfile=LOGS/f"server_{alias}.log"
    fh=open(logfile,"w")

    cmd=[
        VLLM_PY,"-m","vllm.entrypoints.openai.api_server",
        "--model",str(model_path),
        "--served-model-name",alias,
        "--host","127.0.0.1",
        "--port",str(PORT),
        "--gpu-memory-utilization","0.88",
        "--max-model-len",str(CFG["max_model_len"]),
        "--enable-auto-tool-choice",
        "--tool-call-parser",spec["tool_parser"],
    ] + list(spec.get("extra_server_args",[]))

    print("Starting isolated vLLM:", " ".join(map(str,cmd)))
    SERVER=subprocess.Popen(
        cmd,stdout=fh,stderr=subprocess.STDOUT,
        cwd=str(BASE),env=os.environ.copy()
    )

    client=OpenAI(api_key=LOCAL_KEY,base_url=LOCAL_BASE)

    deadline=time.time()+600
    last_error=""
    while time.time()<deadline:
        if SERVER.poll() is not None:
            fh.flush()
            tail=logfile.read_text(errors="ignore")[-7000:]
            raise RuntimeError("vLLM server exited during startup:\n"+tail)
        try:
            models=client.models.list()
            if models.data:
                print("Server ready:",models.data[0].id)
                return client
        except Exception as e:
            last_error=repr(e)
        time.sleep(5)

    raise RuntimeError("Timed out waiting for vLLM server: "+last_error)

def tool_preflight(client,spec):
    result={
        "slug":spec["slug"],"family":spec["family"],"repo":spec["repo"],
        "chat_ok":False,"auto_tool_ok":False,"forced_tool_ok":False,"error":""
    }
    tool=[{
        "type":"function",
        "function":{
            "name":"lookup_order",
            "description":"Look up an order by order ID",
            "parameters":{
                "type":"object",
                "properties":{"order_id":{"type":"string"}},
                "required":["order_id"],
            }
        }
    }]
    try:
        r=client.chat.completions.create(
            model=spec["slug"],
            messages=[{"role":"user","content":"Reply exactly OK."}],
            temperature=0,max_tokens=32,
        )
        result["chat_ok"]=bool(r.choices)
    except Exception as e:
        result["error"]="CHAT: "+repr(e)
        return result

    # Automatic tool choice matters for AgentDojo/tau.
    try:
        r=client.chat.completions.create(
            model=spec["slug"],
            messages=[{"role":"user","content":"You must look up order A123 using the provided tool before answering."}],
            tools=tool,tool_choice="auto",temperature=0,max_tokens=160,
        )
        calls=getattr(r.choices[0].message,"tool_calls",None) if r.choices else None
        result["auto_tool_ok"]=bool(calls and calls[0].function.name=="lookup_order")
    except Exception as e:
        result["error"]+=" AUTO_TOOL: "+repr(e)

    try:
        r=client.chat.completions.create(
            model=spec["slug"],
            messages=[{"role":"user","content":"Call lookup_order for A123."}],
            tools=tool,
            tool_choice={"type":"function","function":{"name":"lookup_order"}},
            temperature=0,max_tokens=160,
        )
        calls=getattr(r.choices[0].message,"tool_calls",None) if r.choices else None
        result["forced_tool_ok"]=bool(calls and calls[0].function.name=="lookup_order")
    except Exception as e:
        result["error"]+=" FORCED_TOOL: "+repr(e)

    return result

print("Model lifecycle helpers ready.")

In [ ]:
# CELL 6 — BENCHMARK RUNNERS + NATIVE PARSERS

def parse_bfcl(root,spec):
    rec=[]
    root=Path(root)
    for p in root.rglob("*"):
        if not p.is_file(): continue
        rel=str(p.relative_to(root))
        try:
            if p.suffix.lower()==".csv":
                df=pd.read_csv(p)
                for c in df.columns:
                    if any(k in str(c).lower() for k in ["accuracy","score"]):
                        for v in pd.to_numeric(df[c],errors="coerce").dropna():
                            rec.append({
                                "benchmark":"BFCL-v4","slug":spec["slug"],"family":spec["family"],
                                "model":spec["repo"],"slice":rel,"metric":str(c),"score":float(v)
                            })
            elif p.suffix.lower() in {".json",".jsonl"}:
                texts=(p.read_text(errors="ignore").splitlines()
                       if p.suffix.lower()==".jsonl" else [p.read_text(errors="ignore")])
                for txt in texts:
                    try:o=json.loads(txt)
                    except Exception:continue
                    stack=[("",o)]
                    while stack:
                        path,x=stack.pop()
                        if isinstance(x,dict):
                            for k,v in x.items():
                                q=f"{path}.{k}" if path else str(k)
                                if isinstance(v,(dict,list)):stack.append((q,v))
                                elif isinstance(v,(int,float)) and any(t in k.lower() for t in ["accuracy","score"]):
                                    rec.append({
                                        "benchmark":"BFCL-v4","slug":spec["slug"],"family":spec["family"],
                                        "model":spec["repo"],"slice":rel,"metric":q,"score":float(v)
                                    })
                        elif isinstance(x,list):
                            for i,v in enumerate(x):stack.append((f"{path}[{i}]",v))
        except Exception:
            pass
    return pd.DataFrame(rec).drop_duplicates() if rec else pd.DataFrame()

def run_bfcl(spec):
    out=RAW/"bfcl"/spec["slug"]
    out.mkdir(parents=True,exist_ok=True)
    runner=WORK/"run_local_bfcl.py"
    runner.write_text(
f'''from evalscope import run_task
from evalscope.config import TaskConfig
cfg=TaskConfig(
 model={spec["slug"]!r},
 api_url={LOCAL_BASE!r},
 api_key={LOCAL_KEY!r},
 eval_type="openai_api",
 datasets=["bfcl_v4"],
 work_dir={str(out)!r},
 limit={CFG["bfcl_limit"]!r},
 seed={SEED},
 generation_config={{"temperature":0.0,"max_tokens":1024,"timeout":180}},
 dataset_args={{"bfcl_v4":{{"extra_params":{{"is_fc_model":True}}}}}}
)
run_task(task_cfg=cfg)
''')
    p=sh([BFPY,runner],cwd=out,timeout=None)
    save_log(f"bfcl_{spec['slug']}.log",p,[BFPY,runner])
    d=parse_bfcl(out,spec)
    status="SUPPORTED" if len(d)>0 else "FAILED"
    return status,d,p.returncode

def parse_dojo(root,spec,suite):
    rec=[]
    for p in Path(root).rglob("*.json"):
        try:o=json.loads(p.read_text())
        except Exception:continue
        if not isinstance(o,dict):continue
        u=o.get("utility");s=o.get("security")
        if not isinstance(u,bool) and not isinstance(s,bool):continue
        rec.append({
            "benchmark":"AgentDojo","slug":spec["slug"],"family":spec["family"],
            "model":spec["repo"],"suite":suite,
            "utility":np.nan if not isinstance(u,bool) else int(u),
            "security":np.nan if not isinstance(s,bool) else int(s),
            "error":o.get("error"),"source_file":str(p),
        })
    return pd.DataFrame(rec)

def run_dojo(spec):
    parts=[]; statuses=[]
    env=os.environ.copy()
    env["OPENAI_COMPATIBLE_BASE_URL"]=LOCAL_BASE
    env["OPENAI_COMPATIBLE_API_KEY"]=LOCAL_KEY
    for suite in CFG["dojo_suites"]:
        out=RAW/"agentdojo"/spec["slug"]/suite
        out.mkdir(parents=True,exist_ok=True)
        cmd=[
            "uv","run","python","-m","agentdojo.scripts.benchmark",
            "--model","openai-compatible","--model-id",spec["slug"],
            "--suite",suite,"--attack","important_instructions",
            "--logdir",str(out),"--force-rerun","--max-workers","1",
        ]
        if CFG["dojo_user_tasks"] is not None:
            for x in CFG["dojo_user_tasks"]:cmd+=["--user-task",x]
        if CFG["dojo_injection_tasks"] is not None:
            for x in CFG["dojo_injection_tasks"]:cmd+=["--injection-task",x]

        p=sh(cmd,cwd=DOJO,env=env,timeout=None)
        save_log(f"dojo_{spec['slug']}_{suite}.log",p,cmd)
        d=parse_dojo(out,spec,suite)
        if len(d):parts.append(d)
        valid=int(((d.utility.notna())|(d.security.notna())).sum()) if len(d) else 0
        errors=int(d.error.notna().sum()) if len(d) else 0
        st="SUPPORTED" if valid>0 and errors==0 else ("PARTIAL" if valid>0 else "FAILED")
        statuses.append({"suite":suite,"status":st,"valid":valid,"errors":errors,"returncode":p.returncode})
    all_cases=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()
    overall="SUPPORTED" if statuses and all(x["status"]=="SUPPORTED" for x in statuses) else ("PARTIAL" if len(all_cases) else "FAILED")
    return overall,all_cases,pd.DataFrame(statuses)

def parse_tau(path,spec,domain):
    try:o=json.loads(Path(path).read_text())
    except Exception:return pd.DataFrame()
    if isinstance(o,list):sims=o
    elif isinstance(o,dict):
        sims=next((o[k] for k in ["simulations","results","trajectories"] if isinstance(o.get(k),list)),[])
    else:sims=[]
    rec=[]
    for i,s in enumerate(sims):
        if not isinstance(s,dict):continue
        reward=None
        if isinstance(s.get("reward_info"),dict) and isinstance(s["reward_info"].get("reward"),(int,float,bool)):
            reward=float(s["reward_info"]["reward"])
        elif isinstance(s.get("reward"),(int,float,bool)):
            reward=float(s["reward"])
        err=s.get("error")
        if err is None and isinstance(s.get("info"),dict):err=s["info"].get("error")
        rec.append({
            "benchmark":"tau3","slug":spec["slug"],"family":spec["family"],"model":spec["repo"],
            "domain":domain,"trajectory_index":i,"task_id":s.get("task_id"),
            "reward":reward,"error":err
        })
    return pd.DataFrame(rec)

def run_tau(spec):
    parts=[];statuses=[]
    env=os.environ.copy()
    # LiteLLM-compatible OpenAI local endpoint.
    env["OPENAI_API_KEY"]=LOCAL_KEY
    env["OPENAI_API_BASE"]=LOCAL_BASE

    for domain in CFG["tau_domains"]:
        run_name=f"ntx_local_{spec['slug']}_{domain}"
        live=TAU/"data"/"simulations"/run_name
        archive=RAW/"tau"/spec["slug"]/domain

        llm_args=json.dumps({
            "api_base":LOCAL_BASE,
            "api_key":LOCAL_KEY,
            "temperature":0.0,
            "max_tokens":1024,
        })

        cmd=[
            "uv","run","tau2","run",
            "--domain",domain,
            "--agent-llm","openai/"+spec["slug"],
            "--user-llm","openai/"+spec["slug"],
            "--agent-llm-args",llm_args,
            "--user-llm-args",llm_args,
            "--num-trials","1",
            "--task-split-name","base",
            "--max-steps",str(CFG["tau_steps"]),
            "--max-errors","3",
            "--max-concurrency","1",
            "--max-retries","1",
            "--retry-delay","2",
            "--seed",str(SEED),
            "--save-to",run_name,
            "--auto-resume","--verbose-logs","--llm-log-mode","all",
        ]
        if CFG["tau_tasks"] is not None:
            cmd+=["--num-tasks",str(CFG["tau_tasks"])]

        p=sh(cmd,cwd=TAU,env=env,timeout=None)
        save_log(f"tau_{spec['slug']}_{domain}.log",p,cmd)

        if live.exists():
            if archive.exists():shutil.rmtree(archive)
            shutil.copytree(live,archive)

        d=parse_tau(archive/"results.json",spec,domain) if (archive/"results.json").exists() else pd.DataFrame()
        if len(d):parts.append(d)
        n=int(d.reward.notna().sum()) if len(d) else 0
        errors=int(d.error.notna().sum()) if len(d) else 0
        st="SUPPORTED" if n>0 and errors==0 else ("PARTIAL" if n>0 else "FAILED")
        statuses.append({"domain":domain,"status":st,"evaluated":n,"errors":errors,"returncode":p.returncode})

    cases=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()
    overall="SUPPORTED" if statuses and all(x["status"]=="SUPPORTED" for x in statuses) else ("PARTIAL" if len(cases) else "FAILED")
    return overall,cases,pd.DataFrame(statuses)

print("Benchmark runners ready.")

In [ ]:
# CELL 7 — RUN ALL GPU-ELIGIBLE MODELS SEQUENTIALLY

model_status=[]
bfcl_all=[]
dojo_all=[]
dojo_slice_status=[]
tau_all=[]
tau_slice_status=[]
successful_preflights=0

for spec in MODEL_REGISTRY:
    if spec["min_vram_gb"]>VRAM_GB:
        model_status.append({
            "slug":spec["slug"],"family":spec["family"],"repo":spec["repo"],
            "download":"SKIPPED","server":"SKIPPED","preflight":"SKIPPED",
            "bfcl":"SKIPPED","agentdojo":"SKIPPED","tau3":"SKIPPED",
            "reason":f"Needs >= {spec['min_vram_gb']} GiB VRAM; GPU has {VRAM_GB:.1f}"
        })
        continue

    if MAX_SUCCESSFUL_MODELS is not None and successful_preflights>=MAX_SUCCESSFUL_MODELS:
        break

    print("\n"+"="*100)
    print("MODEL:",spec["repo"],"| family:",spec["family"])
    print("="*100)

    row={
        "slug":spec["slug"],"family":spec["family"],"repo":spec["repo"],
        "download":"FAILED","server":"FAILED","preflight":"FAILED",
        "bfcl":"NOT_RUN","agentdojo":"NOT_RUN","tau3":"NOT_RUN","reason":""
    }
    local_path=None

    try:
        local_path,revision,size_gb=download_model(spec)
        row["download"]="OK"
        row["revision"]=revision
        row["remote_size_gb"]=round(size_gb,3)
    except Exception as e:
        row["reason"]="DOWNLOAD: "+repr(e)
        model_status.append(row)
        continue

    try:
        client=start_server(spec,local_path)
        row["server"]="OK"
    except Exception as e:
        row["reason"]="SERVER: "+repr(e)
        model_status.append(row)
        stop_server()
        if not KEEP_MODEL_WEIGHTS and local_path:
            shutil.rmtree(local_path,ignore_errors=True)
        continue

    try:
        pf=tool_preflight(client,spec)
        row.update({
            "chat_ok":pf["chat_ok"],
            "auto_tool_ok":pf["auto_tool_ok"],
            "forced_tool_ok":pf["forced_tool_ok"],
        })
        # AgentDojo and tau3 need autonomous tool use, so require auto tool calls.
        if not (pf["chat_ok"] and pf["auto_tool_ok"]):
            row["reason"]="TOOL_PREFLIGHT: "+pf["error"]
            model_status.append(row)
            stop_server()
            if not KEEP_MODEL_WEIGHTS and local_path:
                shutil.rmtree(local_path,ignore_errors=True)
            continue
        row["preflight"]="OK"
        successful_preflights+=1
    except Exception as e:
        row["reason"]="PREFLIGHT: "+repr(e)
        model_status.append(row)
        stop_server()
        if not KEEP_MODEL_WEIGHTS and local_path:
            shutil.rmtree(local_path,ignore_errors=True)
        continue

    # BFCL
    try:
        st,d,rc=run_bfcl(spec)
        row["bfcl"]=st
        row["bfcl_returncode"]=rc
        if len(d):bfcl_all.append(d)
    except Exception as e:
        row["bfcl"]="FAILED"
        row["reason"]+=" BFCL:"+repr(e)

    # AgentDojo
    try:
        st,d,ss=run_dojo(spec)
        row["agentdojo"]=st
        if len(d):dojo_all.append(d)
        if len(ss):
            ss["slug"]=spec["slug"];ss["family"]=spec["family"];dojo_slice_status.append(ss)
    except Exception as e:
        row["agentdojo"]="FAILED"
        row["reason"]+=" DOJO:"+repr(e)

    # tau3
    try:
        st,d,ss=run_tau(spec)
        row["tau3"]=st
        if len(d):tau_all.append(d)
        if len(ss):
            ss["slug"]=spec["slug"];ss["family"]=spec["family"];tau_slice_status.append(ss)
    except Exception as e:
        row["tau3"]="FAILED"
        row["reason"]+=" TAU:"+repr(e)

    model_status.append(row)

    # Save after each model so a disconnected runtime still leaves evidence.
    pd.DataFrame(model_status).to_csv(RESULTS/"10_model_run_status.csv",index=False)

    stop_server()

    if not KEEP_MODEL_WEIGHTS and local_path:
        shutil.rmtree(local_path,ignore_errors=True)
        gc.collect()

stop_server()

status_df=pd.DataFrame(model_status)
display(status_df)
status_df.to_csv(RESULTS/"10_model_run_status.csv",index=False)

bfcl_df=pd.concat(bfcl_all,ignore_index=True) if bfcl_all else pd.DataFrame()
dojo_df=pd.concat(dojo_all,ignore_index=True) if dojo_all else pd.DataFrame()
tau_df=pd.concat(tau_all,ignore_index=True) if tau_all else pd.DataFrame()
dojo_slices=pd.concat(dojo_slice_status,ignore_index=True) if dojo_slice_status else pd.DataFrame()
tau_slices=pd.concat(tau_slice_status,ignore_index=True) if tau_slice_status else pd.DataFrame()

bfcl_df.to_csv(RESULTS/"20_bfcl_native_metrics.csv",index=False)
dojo_df.to_csv(RESULTS/"21_agentdojo_native_cases.csv",index=False)
dojo_slices.to_csv(RESULTS/"21_agentdojo_slice_status.csv",index=False)
tau_df.to_csv(RESULTS/"22_tau3_native_cases.csv",index=False)
tau_slices.to_csv(RESULTS/"22_tau3_slice_status.csv",index=False)

In [ ]:
# CELL 8 — AGGREGATE BENCHMARK-NATIVE RESULTS + PAPER CLOSURE GATE

summary_rows=[]

if len(bfcl_df):
    for (slug,family,metric),g in bfcl_df.groupby(["slug","family","metric"]):
        summary_rows.append({
            "benchmark":"BFCL-v4","slug":slug,"family":family,"metric":metric,
            "n":len(g),"mean":float(g.score.mean())
        })

if len(dojo_df):
    for (slug,family),g in dojo_df.groupby(["slug","family"]):
        if g.utility.notna().any():
            summary_rows.append({
                "benchmark":"AgentDojo","slug":slug,"family":family,"metric":"utility",
                "n":int(g.utility.notna().sum()),"mean":float(g.utility.mean())
            })
        if g.security.notna().any():
            summary_rows.append({
                "benchmark":"AgentDojo","slug":slug,"family":family,"metric":"security",
                "n":int(g.security.notna().sum()),"mean":float(g.security.mean())
            })

if len(tau_df):
    for (slug,family),g in tau_df.groupby(["slug","family"]):
        summary_rows.append({
            "benchmark":"tau3","slug":slug,"family":family,"metric":"reward",
            "n":int(g.reward.notna().sum()),"mean":float(g.reward.mean())
        })

summary=pd.DataFrame(summary_rows)
summary.to_csv(RESULTS/"30_external_model_summary.csv",index=False)
display(summary)

# A family counts as complete only if at least one model from that family
# has SUPPORTED status for all three benchmark families.
complete_models=status_df[
    (status_df["bfcl"]=="SUPPORTED")&
    (status_df["agentdojo"]=="SUPPORTED")&
    (status_df["tau3"]=="SUPPORTED")
].copy()

complete_families=sorted(set(complete_models["family"].astype(str)))
closure_supported=len(complete_families)>=3

claims=pd.DataFrame([
    {
        "claim":"Local open-weight BFCL-v4 evidence",
        "status":"SUPPORTED" if (status_df["bfcl"]=="SUPPORTED").any() else "MISSING",
    },
    {
        "claim":"Local open-weight AgentDojo evidence",
        "status":"SUPPORTED" if (status_df["agentdojo"]=="SUPPORTED").any() else "MISSING",
    },
    {
        "claim":"Local open-weight tau3 evidence",
        "status":"SUPPORTED" if (status_df["tau3"]=="SUPPORTED").any() else "MISSING",
    },
    {
        "claim":"At least 3 independent local model families complete all benchmarks",
        "status":"SUPPORTED" if closure_supported else "INCOMPLETE",
    },
    {
        "claim":"External local-model paper closure gate",
        "status":"SUPPORTED" if closure_supported else "INCOMPLETE",
    },
])

claims.to_csv(RESULTS/"31_claim_gate.csv",index=False)
display(claims)

print("Complete families:",complete_families)

In [ ]:
# CELL 9 — PAPER TABLES / FIGURES / PROVENANCE

summary.to_latex(
    PAPER/"local_external_summary.tex",
    index=False,float_format="%.4f"
)
claims.to_latex(
    PAPER/"local_external_claim_gate.tex",
    index=False
)
status_df.to_csv(PAPER/"local_model_status.csv",index=False)

if len(summary):
    for (benchmark,metric),g in summary.groupby(["benchmark","metric"]):
        gg=g.sort_values("mean")
        fig,ax=plt.subplots(figsize=(8,max(3,0.45*len(gg)+1)))
        ax.barh(gg["slug"],gg["mean"])
        ax.set_title(f"{benchmark}: {metric}")
        ax.set_xlabel(metric)
        fig.tight_layout()
        safe=re.sub(r"[^A-Za-z0-9]+","_",f"{benchmark}_{metric}")
        fig.savefig(PAPER/f"{safe}.png",dpi=220,bbox_inches="tight")
        plt.show()

if closure_supported:
    section=(
        "\\paragraph{External local-model validation.}\n"
        "We additionally evaluated the runtime using locally served open-weight models "
        "without relying on commercial inference APIs. At least three independent model "
        f"families completed BFCL-v4, AgentDojo, and $\\tau^3$: {', '.join(complete_families)}. "
        "Each model was served independently through a local OpenAI-compatible vLLM endpoint, "
        "and only native benchmark outputs with evaluated cases were retained."
    )
else:
    section=(
        "\\paragraph{External local-model validation.}\n"
        "We evaluated a set of locally served open-weight models on BFCL-v4, AgentDojo, "
        "and $\\tau^3$. Because fewer than three independent model families completed all "
        "three benchmark families, these results are reported as bounded external evidence "
        "rather than a complete cross-family generalization claim."
    )

(PAPER/"local_external_validation_section.tex").write_text(section+"\n")

# Model/revision provenance.
manifest={
    "experiment":"NTX-LOCAL-OPEN-WEIGHT-EXTERNAL-CLOSURE",
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "mode":MODE,
    "gpu_name":GPU_NAME,
    "gpu_vram_gb":VRAM_GB,
    "benchmark_versions":BENCHMARK_VERSIONS,
    "complete_families":complete_families,
    "closure_supported":closure_supported,
    "model_status":status_df.to_dict("records"),
    "config":CFG,
}
(RESULTS/"FINAL_MANIFEST.json").write_text(json.dumps(manifest,indent=2,default=str))

# Hash research outputs (not giant model weights).
hash_rows=[]
for label,folder in [("results",RESULTS),("paper",PAPER),("logs",LOGS)]:
    for p in sorted(folder.rglob("*")):
        if p.is_file() and p.name!="SHA256_MANIFEST.csv":
            h=hashlib.sha256()
            with open(p,"rb") as f:
                for chunk in iter(lambda:f.read(1024*1024),b""):h.update(chunk)
            hash_rows.append({"file":f"{label}/{p.relative_to(folder)}","sha256":h.hexdigest()})
pd.DataFrame(hash_rows).to_csv(RESULTS/"SHA256_MANIFEST.csv",index=False)

In [ ]:
# CELL 10 — ZIP ALL RESULTS AND DOWNLOAD

stage=BASE/"final_package"
if stage.exists():shutil.rmtree(stage)
stage.mkdir()

for src,name in [
    (RESULTS,"results"),
    (RAW,"raw_benchmark_outputs"),
    (LOGS,"logs"),
    (PAPER,"paper_integration"),
]:
    if src.exists():
        shutil.copytree(src,stage/name)

# Small reproducibility files only; do not package multi-GB model weights.
(stage/"README.txt").write_text(
    "NiyamTrace-X local open-weight external validation package.\n"
    "Model weights are intentionally excluded. See results/FINAL_MANIFEST.json "
    "and results/10_model_run_status.csv for model IDs/revisions/status.\n"
)

ZIP=ARCH/"NTX_LOCAL_MODELS_EXTERNAL_CLOSURE_RESULTS.zip"
if ZIP.exists():ZIP.unlink()

with zipfile.ZipFile(ZIP,"w",zipfile.ZIP_DEFLATED,allowZip64=True) as z:
    for p in stage.rglob("*"):
        if p.is_file():
            z.write(p,arcname=str(p.relative_to(stage)))

with zipfile.ZipFile(ZIP) as z:
    bad=z.testzip()

h=hashlib.sha256()
with open(ZIP,"rb") as f:
    for chunk in iter(lambda:f.read(1024*1024),b""):h.update(chunk)

print("Closure gate:", "SUPPORTED" if closure_supported else "INCOMPLETE")
print("Complete families:",complete_families)
print("ZIP integrity:", "PASS" if bad is None else bad)
print("ZIP:",ZIP)
print("SHA256:",h.hexdigest())

try:
    from google.colab import files
    files.download(str(ZIP))
except Exception as e:
    print("Auto-download unavailable:",repr(e))